# Dynamic Knowledge Updating in CAG — Colab benchmark (SMOKE profile)

Runs the whole pipeline on a single GPU (T4 16 GB is enough; tested locally on a 4 GB GTX 1650):
dataset preparation → synthetic updates/QA → exact-cache validation → 7 methods (B0–B5 + approximate B3a) → metrics → figures.

**Runtime → Change runtime type → T4 GPU** before running. All numbers you see are produced by this run; nothing is precomputed.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. Get the project
Either upload `cag-benchmark.zip` (the repository folder, without `.venv`, `models`, `.wheels`) to the Colab file browser, or mount Google Drive where the folder lives.

In [ ]:
import os, zipfile
USE_DRIVE = False  # set True if the folder is in Google Drive
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/cag-benchmark'
else:
    PROJECT = '/content/cag-benchmark'
    if not os.path.exists(PROJECT) and os.path.exists('/content/cag-benchmark.zip'):
        zipfile.ZipFile('/content/cag-benchmark.zip').extractall('/content')
assert os.path.exists(PROJECT), 'upload cag-benchmark.zip or set USE_DRIVE'
%cd {PROJECT}

## 2. Install dependencies
Colab already ships CUDA PyTorch; only the remaining packages are installed.

In [ ]:
!pip -q install -r requirements-colab.txt
import torch, transformers
print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), transformers.__version__)
assert torch.cuda.is_available(), 'No GPU: use run_smoke.py --device cpu --allow-cpu for a labelled CPU smoke test'

## 3. Dataset
The source export must be at `data/raw/database_export_2026-01-23T02-02-46.json` (it is never modified; its SHA-256 is recorded).

In [ ]:
RAW = 'data/raw/database_export_2026-01-23T02-02-46.json'
if not os.path.exists(RAW):
    from google.colab import files
    up = files.upload()
    os.makedirs('data/raw', exist_ok=True)
    os.replace(list(up)[0], RAW)
!python scripts/prepare_dataset.py
import json; prof = json.load(open('data/processed/dataset_profile.json', encoding='utf-8'))
print(prof['record_counts']); print(prof['documents_by_kind']); print('tokens:', prof['tokens_by_kind'])

## 4. Synthetic updates and evaluation questions

In [ ]:
!python scripts/generate_benchmark.py --config configs/smoke.yaml
print(json.load(open('data/benchmark/smoke/coverage.json', encoding='utf-8')))

## 5. Unit tests (data, evaluators, cache logic on a tiny fp32 model)

In [ ]:
!python -m pytest -q tests

## 6. Exact-cache validation on the real model (10 mandated update scenarios)
B3/B4/B5 must match one-shot Full Recompute within the dtype tolerance; B3a is approximate and is expected to deviate.

In [ ]:
!python scripts/validate_cache.py --config configs/smoke.yaml --budget 1024 --out results/smoke/cache_validation.json

## 7. Benchmark (SMOKE): build initial caches, apply updates, answer questions, measure

In [ ]:
!python scripts/run_benchmark.py --config configs/smoke.yaml --out results/smoke

## 8. Metrics, tables and figures

In [ ]:
!python scripts/generate_report.py --results results/smoke
import pandas as pd
s = pd.read_csv('results/smoke/benchmark_summary.csv')
cols = ['baseline','scenario','update_latency_s_mean','query_latency_s_mean','updated_fact_accuracy',
        'stale_answer_rate','unchanged_fact_accuracy','cache_reuse_ratio','peak_gpu_allocated_mb','validation_status']
s.groupby('baseline')[cols[2:-1]].mean()

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('results/smoke/figures/*.png')):
    print(p); display(Image(p, width=640))

## 9. Export

In [ ]:
!zip -qr smoke_results.zip results/smoke reports data/benchmark/smoke
try:
    from google.colab import files; files.download('smoke_results.zip')
except Exception as e:
    print('download manually:', e)